In [0]:
%sql
CREATE OR REPLACE TABLE retail_lakehouse.gold.daily_sales_summary AS
SELECT
  order_date,
  COUNT(DISTINCT order_id) AS total_orders,
  SUM(quantity) AS total_quantity,
  SUM(total_amount) AS total_sales
FROM retail_lakehouse.silver.sales
GROUP BY order_date;

In [0]:
%sql
SELECT * 
FROM retail_lakehouse.gold.daily_sales_summary;

In [0]:
%sql
CREATE OR REPLACE TABLE retail_lakehouse.gold.monthly_sales_summary AS
SELECT
  date_format(order_date, 'yyyy-MM') AS sales_month,
  COUNT(DISTINCT order_id) AS total_orders,
  SUM(quantity) AS total_quantity,
  SUM(total_amount) AS total_sales
FROM retail_lakehouse.silver.sales
GROUP BY date_format(order_date, 'yyyy-MM');

In [0]:
%sql
SELECT *
FROM retail_lakehouse.gold.monthly_sales_summary;

In [0]:
from datetime import datetime
import uuid

run_id = str(uuid.uuid4())
start_time = datetime.now()
notebook_name = "03_Gold_Aggregation"
layer_name = "gold"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, LongType

end_time = datetime.now()

records_count = spark.sql("""
SELECT COUNT(*) AS cnt
FROM retail_lakehouse.gold.daily_sales_summary
""").collect()[0]["cnt"]

schema = StructType([
    StructField("run_id", StringType(), True),
    StructField("pipeline_id", StringType(), True),
    StructField("layer_name", StringType(), True),
    StructField("status", StringType(), True),
    StructField("start_time", TimestampType(), True),
    StructField("end_time", TimestampType(), True),
    StructField("records_read", LongType(), True),
    StructField("records_written", LongType(), True),
    StructField("error_message", StringType(), True)
])

audit_df = spark.createDataFrame([{
    "run_id": run_id,
    "pipeline_id": notebook_name,
    "layer_name": layer_name,
    "status": "SUCCESS",
    "start_time": start_time,
    "end_time": end_time,
    "records_read": 0,
    "records_written": records_count,
    "error_message": None
}], schema=schema)

audit_df.write.mode("append").saveAsTable("retail_lakehouse.audit.pipeline_audit")

In [0]:
%sql
SELECT *
FROM retail_lakehouse.audit.pipeline_audit
ORDER BY start_time DESC;